# Melhoria e Validação do Modelo de Predição de Aluguéis

## Objetivo

Este notebook tem como objetivo investigar estratégias para melhorar o desempenho do modelo de predição de aluguéis desenvolvido no projeto.

Na etapa anterior de modelagem (05_modeling.ipynb), foram avaliados diferentes algoritmos de regressão utilizando as características originais dos imóveis. O `XGBoost Regressor` apresentou o melhor desempenho inicial, embora a diferença em relação ao `Gradient Boosting Regressor` tenha sido pequena.

Dessa forma, nesta etapa o XGBoost será utilizado como modelo principal para investigar se as features desenvolvidas durante a etapa de Feature Engineering (04_feature_engineering.ipynb) contribuem para melhorar a capacidade preditiva do modelo. Além da comparação entre diferentes conjuntos de features, será introduzida a validação cruzada K-Fold, permitindo avaliar de forma mais robusta a estabilidade do modelo em diferentes subconjuntos dos dados de treinamento.

---

## Estratégia experimental

Os experimentos serão realizados de forma progressiva, permitindo comparar o desempenho do modelo base com diferentes configurações de Feature Engineering.

Serão avaliadas quatro configurações principais:

### Experimento 1 — Baseline

O primeiro experimento utilizará apenas as características originais selecionadas para a modelagem:

- `area`;
- `bedrooms`;
- `garage`;
- `type`.

Este experimento servirá como referência para avaliar o impacto das novas features.

### Experimento 2 — Standard Score Global

Serão adicionadas as features:

- `standard_score`;
- `high_standard`.

O objetivo é avaliar se uma medida global de porte do imóvel e uma classificação empírica de alto padrão fornecem informações adicionais ao modelo.

### Experimento 3 — Standard Score por Tipo

Serão adicionadas as features:

- `standard_score_type`;
- `high_standard_type`.

Neste experimento, o porte relativo do imóvel será avaliado dentro de cada categoria de imóvel, permitindo identificar imóveis que apresentam características de maior padrão em relação a outros imóveis do mesmo tipo.

---

## Validação dos experimentos

Os experimentos serão avaliados utilizando validação cruzada K-Fold aplicada ao conjunto de treinamento. A validação cruzada permitirá avaliar o comportamento do modelo em diferentes subconjuntos dos dados, reduzindo a dependência de uma única divisão entre treinamento e validação.

Para cada experimento serão analisadas as seguintes métricas:

- **MAE (Mean Absolute Error)**: representa o erro absoluto médio das previsões, sendo interpretado diretamente na unidade monetária do aluguel;
- **RMSE (Root Mean Squared Error)**: penaliza mais fortemente erros de maior magnitude;
- **R² (Coefficient of Determination)**: representa a proporção da variabilidade da variável target explicada pelo modelo.

Além da média das métricas obtidas nos folds, será analisado o desvio padrão dos resultados, permitindo avaliar a estabilidade do modelo.

O conjunto de teste será mantido separado durante todo o processo de validação e comparação dos experimentos. Após a seleção da melhor configuração, o modelo final será treinado utilizando o conjunto de treinamento e avaliado uma única vez no conjunto de teste independente.

---

## Fluxo metodológico

A metodologia adotada neste notebook seguirá o fluxo:

1. Carregamento dos dados;
2. Definição das variáveis de modelagem;
3. Divisão entre treinamento e teste;
4. Definição do pré-processamento;
5. Definição do modelo XGBoost;
6. Definição da validação cruzada K-Fold;
7. Execução do experimento baseline;
8. Execução dos experimentos com Feature Engineering;
9. Comparação dos resultados;
10. Seleção da melhor configuração;
11. Treinamento do modelo final;
12. Avaliação no conjunto de teste;
13. Análise dos erros do modelo final.

O objetivo final é verificar se as features desenvolvidas durante a etapa de Feature Engineering proporcionam uma melhoria consistente em relação ao baseline inicial do XGBoost.

## Importação de bibliotecas e Carregamento dos dados

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from xgboost import XGBRegressor

In [2]:
# Carregar dados limpos
df = pd.read_csv("../data/processed/data_cleaned.csv")

# Separar features que não entrarão no modelo
X = df.drop(
    columns=[
        "rent",
        "total",
        "address",
        "district"
    ]
)

# Target
y = df["rent"]

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

## Pré-processamento

In [4]:
numeric_features = [
    "area",
    "bedrooms",
    "garage"
]

categorical_features = [
    "type"
]

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [6]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

In [7]:
baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

In [8]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## 01 - XGBoost Baseline com 5-Fold

In [9]:
cv_results = cross_validate(
    estimator=baseline_pipeline,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    return_train_score=False,
    n_jobs=-1
)

In [10]:
cv_results_df = pd.DataFrame({
    "MAE": -cv_results["test_MAE"],
    "RMSE": -cv_results["test_RMSE"],
    "R2": cv_results["test_R2"]
})

cv_results_df

,MAE,RMSE,R2
0,1154.535522,1806.485107,0.580081
1,1106.982178,1759.733398,0.556669
2,1150.785767,1819.287598,0.507736
3,1121.381836,1775.944824,0.541596
4,1085.265991,1735.954956,0.543981


In [11]:
cv_mean = cv_results_df.mean()

cv_mean

MAE     1123.790259
RMSE    1779.481177
R2         0.546013
dtype: float64

In [12]:
cv_std = cv_results_df.std()

cv_std

MAE     29.353279
RMSE    33.951347
R2       0.026282
dtype: float64

In [13]:
baseline_results = pd.DataFrame({
    "Model": ["XGBoost Baseline"],
    "MAE_mean": [cv_results_df["MAE"].mean()],
    "MAE_std": [cv_results_df["MAE"].std()],
    "RMSE_mean": [cv_results_df["RMSE"].mean()],
    "RMSE_std": [cv_results_df["RMSE"].std()],
    "R2_mean": [cv_results_df["R2"].mean()],
    "R2_std": [cv_results_df["R2"].std()]
})

baseline_results

,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,XGBoost Baseline,1123.790259,29.353279,1779.481177,33.951347,0.546013,0.026282


O primeiro experimento do notebook utilizou o XGBoost com as características originais `area`, `bedrooms`, `garage` e `type`. O modelo foi avaliado utilizando validação cruzada K-Fold com 5 folds sobre o conjunto de treinamento. O pré-processamento foi integrado ao pipeline, garantindo que o StandardScaler e o OneHotEncoder fossem ajustados independentemente em cada fold.

O modelo apresentou MAE médio de R$ 1.123,79, RMSE médio de R$ 1.779,48 e R² médio de 0,5460. Os desvios padrão foram de R$ 29,35 para o MAE, R$ 33,95 para o RMSE e 0,0263 para o R², indicando uma variação relativamente pequena entre os folds.

Esses resultados serão utilizados como baseline para avaliar o impacto das features desenvolvidas durante a etapa de Feature Engineering.

## 02 - XGBoost com StandardScore global

### Criação das variáveis auxiliares

O `standard_score` global é definido como:

$$
Z = \frac{X-\mu}{\sigma}
$$

e então calcular:

$$
standard\_score =
Z_{area} +
Z_{bedrooms} +
Z_{garage}
$$

Posteriormente, essa pontuação poderá ser utilizada para criar uma variável categórica ou binária, denominada `high_standard`, classificando os imóveis com maior padrão estrutural.

In [36]:
class HighStandardTransformer(
    BaseEstimator,
    TransformerMixin
):

    def __init__(self, quantile=0.90):
        self.quantile = quantile

    def fit(self, X, y=None):

        X = X.copy()

        self.standard_features_ = [
            "area",
            "bedrooms",
            "garage"
        ]

        # Ajusta o scaler APENAS nos dados
        # de treinamento do fold
        self.scaler_ = StandardScaler()

        X_scaled = self.scaler_.fit_transform(
            X[self.standard_features_]
        )

        # Converte para DataFrame para manter
        # a correspondência entre as colunas
        X_scaled = pd.DataFrame(
            X_scaled,
            columns=[
                "area_z",
                "bedrooms_z",
                "garage_z"
            ],
            index=X.index
        )

        # Calcula o standard_score
        standard_score = (
            X_scaled["area_z"]
            + X_scaled["bedrooms_z"]
            + X_scaled["garage_z"]
        )

        # Aprende o threshold APENAS no treino
        self.threshold_ = (
            standard_score
            .quantile(self.quantile)
        )

        return self

    def transform(self, X):

        X = X.copy()

        # Utiliza o scaler aprendido no fit()
        X_scaled = self.scaler_.transform(
            X[self.standard_features_]
        )

        X_scaled = pd.DataFrame(
            X_scaled,
            columns=[
                "area_z",
                "bedrooms_z",
                "garage_z"
            ],
            index=X.index
        )

        # Cria standard_score apenas como
        # variável intermediária
        standard_score = (
            X_scaled["area_z"]
            + X_scaled["bedrooms_z"]
            + X_scaled["garage_z"]
        )

        # Cria a feature final
        X["high_standard"] = (
            standard_score
            >= self.threshold_
        ).astype(int)

        return X

In [37]:
high_standard_transformer = HighStandardTransformer()

X_train_exp2 = (
    high_standard_transformer
    .fit_transform(X_train)
)

In [38]:
X_train_exp2.head()

,area,bedrooms,garage,type,high_standard
6385,46,2,1,Apartamento,0
9894,223,4,2,Casa,1
3031,67,3,1,Apartamento,0
5566,82,3,1,Apartamento,0
6326,87,2,0,Apartamento,0


In [39]:
numeric_features_exp2 = [
    "area",
    "bedrooms",
    "garage"
]

binary_features_exp2 = [
    "high_standard"
]

categorical_features_exp2 = [
    "type"
]

In [40]:
preprocessor_exp2 = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features_exp2
        ),
        (
            "binary",
            "passthrough",
            binary_features_exp2
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features_exp2
        )
    ]
)

In [41]:
xgb_model_exp2 = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

In [42]:
experiment_2_pipeline = Pipeline(
    steps=[
        (
            "high_standard",
            HighStandardTransformer(
                quantile=0.90
            )
        ),
        (
            "preprocessor",
            preprocessor_exp2
        ),
        (
            "model",
            xgb_model_exp2
        )
    ]
)

In [43]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [44]:
cv_results_exp2 = cross_validate(
    estimator=experiment_2_pipeline,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    return_train_score=False,
    n_jobs=-1
)

In [45]:
cv_results_exp2_df = pd.DataFrame({
    "MAE": -cv_results_exp2["test_MAE"],
    "RMSE": -cv_results_exp2["test_RMSE"],
    "R2": cv_results_exp2["test_R2"]
})

cv_results_exp2_df

,MAE,RMSE,R2
0,1154.122925,1810.628540,0.578153
1,1116.070923,1788.464478,0.542075
2,1140.401245,1816.055542,0.509484
3,1124.941284,1781.739502,0.538600
4,1077.988037,1726.268799,0.549056


In [46]:
experiment_2_results = pd.DataFrame({
    "Model": ["XGBoost + High Standard"],
    "MAE_mean": [
        cv_results_exp2_df["MAE"].mean()
    ],
    "MAE_std": [
        cv_results_exp2_df["MAE"].std()
    ],
    "RMSE_mean": [
        cv_results_exp2_df["RMSE"].mean()
    ],
    "RMSE_std": [
        cv_results_exp2_df["RMSE"].std()
    ],
    "R2_mean": [
        cv_results_exp2_df["R2"].mean()
    ],
    "R2_std": [
        cv_results_exp2_df["R2"].std()
    ]
})

experiment_2_results

,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,XGBoost + High Standard,1122.704883,28.934676,1784.631372,35.681146,0.543473,0.02457


### Experimento 2 — XGBoost + `high_standard`

No segundo experimento, foi adicionada a variável `high_standard`, construída a partir do `standard_score` global, calculado com base nas variáveis `area`, `bedrooms` e `garage`.

O `standard_score` foi utilizado apenas como uma variável intermediária para definir o limiar correspondente ao quantil de 90%. Após a criação de `high_standard`, o `standard_score` foi descartado e não foi utilizado diretamente como variável de entrada do modelo.

A criação dessas variáveis foi incorporada ao processo de validação cruzada, garantindo que os parâmetros necessários para o cálculo dos *z-scores* e o limiar de `high_standard` fossem aprendidos exclusivamente a partir dos dados de treinamento de cada *fold*.

#### Resultados

| Model | MAE_mean | MAE_std | RMSE_mean | RMSE_std | R²_mean | R²_std |
|---|---:|---:|---:|---:|---:|---:|
| XGBoost Baseline | 1123,79 | 29,35 | 1779,48 | 33,95 | 0,5460 | 0,0263 |
| XGBoost + High Standard | 1122,70 | 28,93 | 1784,63 | 35,68 | 0,5435 | 0,0246 |

Comparado ao modelo baseline, o modelo com `high_standard` apresentou uma redução muito pequena no MAE médio, de aproximadamente **R$ 1,09**. Entretanto, o RMSE aumentou aproximadamente **R$ 5,15**, enquanto o R² apresentou uma pequena redução de aproximadamente **0,0025**.

De maneira geral, os resultados permaneceram muito próximos aos obtidos pelo modelo baseline. Portanto, nesta etapa, a inclusão de `high_standard` **não apresentou uma melhoria relevante no desempenho preditivo do XGBoost**.

Entretanto, esse resultado não permite concluir que a abordagem de identificação de imóveis de alto padrão seja necessariamente ineficaz. A variável `high_standard` representa uma classificação global dos imóveis e pode não considerar adequadamente as diferenças existentes entre os diferentes tipos de imóveis. Essa hipótese será investigada no próximo experimento, por meio da criação da variável `high_standard_type`, que considera o padrão relativo dos imóveis dentro de cada categoria de `type`.

## 03 - XGBoost com standard score invidual por tipo de imóvel

In [47]:
class HighStandardTypeTransformer(
    BaseEstimator,
    TransformerMixin
):

    def __init__(self, quantile=0.90):
        self.quantile = quantile

    def fit(self, X, y=None):

        X = X.copy()

        self.standard_features_ = [
            "area",
            "bedrooms",
            "garage"
        ]

        self.types_ = X["type"].unique()

        self.scalers_ = {}

        self.thresholds_ = {}

        for property_type in self.types_:

            mask = X["type"] == property_type

            X_type = X.loc[
                mask,
                self.standard_features_
            ]

            scaler = StandardScaler()

            X_type_scaled = scaler.fit_transform(
                X_type
            )

            X_type_scaled = pd.DataFrame(
                X_type_scaled,
                columns=[
                    "area_z",
                    "bedrooms_z",
                    "garage_z"
                ],
                index=X_type.index
            )

            standard_score_type = (
                X_type_scaled["area_z"]
                + X_type_scaled["bedrooms_z"]
                + X_type_scaled["garage_z"]
            )

            threshold = (
                standard_score_type
                .quantile(self.quantile)
            )

            self.scalers_[property_type] = scaler

            self.thresholds_[property_type] = threshold

        return self

    def transform(self, X):

        X = X.copy()

        X["high_standard_type"] = 0

        for property_type in self.types_:

            mask = X["type"] == property_type

            X_type = X.loc[
                mask,
                self.standard_features_
            ]

            scaler = self.scalers_[
                property_type
            ]

            threshold = self.thresholds_[
                property_type
            ]

            X_type_scaled = scaler.transform(
                X_type
            )

            X_type_scaled = pd.DataFrame(
                X_type_scaled,
                columns=[
                    "area_z",
                    "bedrooms_z",
                    "garage_z"
                ],
                index=X_type.index
            )

            standard_score_type = (
                X_type_scaled["area_z"]
                + X_type_scaled["bedrooms_z"]
                + X_type_scaled["garage_z"]
            )

            X.loc[
                mask,
                "high_standard_type"
            ] = (
                standard_score_type
                >= threshold
            ).astype(int)

        return X

In [48]:
high_standard_type_transformer = (
    HighStandardTypeTransformer()
)

In [49]:
X_train_exp3 = (
    high_standard_type_transformer
    .fit_transform(X_train)
)

In [50]:
X_train_exp3.head()

,area,bedrooms,garage,type,high_standard_type
6385,46,2,1,Apartamento,0
9894,223,4,2,Casa,0
3031,67,3,1,Apartamento,0
5566,82,3,1,Apartamento,0
6326,87,2,0,Apartamento,0


In [51]:
numeric_features_exp3 = [
    "area",
    "bedrooms",
    "garage"
]

binary_features_exp3 = [
    "high_standard_type"
]

categorical_features_exp3 = [
    "type"
]

In [52]:
preprocessor_exp3 = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features_exp3
        ),
        (
            "binary",
            "passthrough",
            binary_features_exp3
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features_exp3
        )
    ]
)

In [53]:
xgb_model_exp3 = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

In [54]:
experiment_3_pipeline = Pipeline(
    steps=[
        (
            "high_standard_type",
            HighStandardTypeTransformer(
                quantile=0.90
            )
        ),
        (
            "preprocessor",
            preprocessor_exp3
        ),
        (
            "model",
            xgb_model_exp3
        )
    ]
)

In [55]:
cv_results_exp3 = cross_validate(
    estimator=experiment_3_pipeline,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    return_train_score=False,
    n_jobs=-1
)

In [56]:
cv_results_exp3_df = pd.DataFrame({
    "MAE": -cv_results_exp3["test_MAE"],
    "RMSE": -cv_results_exp3["test_RMSE"],
    "R2": cv_results_exp3["test_R2"]
})

cv_results_exp3_df

,MAE,RMSE,R2
0,1171.809204,1847.057983,0.561007
1,1112.747925,1783.123657,0.544806
2,1139.429199,1803.634521,0.516170
3,1125.915283,1785.168335,0.536823
4,1072.920654,1731.103516,0.546526


In [57]:
experiment_3_results = pd.DataFrame({
    "Model": [
        "XGBoost + High Standard Type"
    ],
    "MAE_mean": [
        cv_results_exp3_df["MAE"].mean()
    ],
    "MAE_std": [
        cv_results_exp3_df["MAE"].std()
    ],
    "RMSE_mean": [
        cv_results_exp3_df["RMSE"].mean()
    ],
    "RMSE_std": [
        cv_results_exp3_df["RMSE"].std()
    ],
    "R2_mean": [
        cv_results_exp3_df["R2"].mean()
    ],
    "R2_std": [
        cv_results_exp3_df["R2"].std()
    ]
})

experiment_3_results

,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,XGBoost + High Standard Type,1124.564453,36.26832,1790.017603,41.776001,0.541066,0.016426


In [58]:
comparison_exp1_exp2_exp3 = pd.concat(
    [
        baseline_results,
        experiment_2_results,
        experiment_3_results
    ],
    ignore_index=True
)

comparison_exp1_exp2_exp3

,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,XGBoost Baseline,1123.790259,29.353279,1779.481177,33.951347,0.546013,0.026282
1,XGBoost + High Standard,1122.704883,28.934676,1784.631372,35.681146,0.543473,0.024570
2,XGBoost + High Standard Type,1124.564453,36.268320,1790.017603,41.776001,0.541066,0.016426


## Conclusão dos experimentos de melhoria

Nesta etapa, foi realizada uma avaliação do modelo `XGBoost` utilizando validação cruzada com `K-Fold`, com o objetivo de obter uma estimativa mais robusta do desempenho do modelo e verificar se as variáveis desenvolvidas durante a etapa de Feature Engineering contribuíam para a melhoria da capacidade preditiva.

O modelo `XGBoost Baseline`, utilizando as características originais `area`, `bedrooms`, `garage` e `type`, apresentou os seguintes resultados médios na validação cruzada:

| Modelo | MAE médio | RMSE médio | R² médio |
|---|---:|---:|---:|
| XGBoost Baseline | R$ 1.123,79 | R$ 1.779,48 | 0,5460 |

A partir desse resultado, foram realizados dois experimentos adicionais.

No primeiro, foi adicionada a variável `high_standard`, criada a partir do `standard_score` global. O modelo apresentou MAE médio de R$ 1.122,70, RMSE médio de R$ 1.784,63 e R² médio de 0,5435. Embora tenha apresentado uma pequena redução no MAE em relação ao baseline, o RMSE e o R² apresentaram uma pequena piora. Portanto, a inclusão de `high_standard` não produziu uma melhoria consistente no desempenho geral do modelo.

No segundo experimento, foi adicionada a variável `high_standard_type`, construída a partir do `standard_score_type`, considerando a posição relativa do imóvel dentro de sua própria categoria de `type`. O modelo apresentou MAE médio de R$ 1.124,56, RMSE médio de R$ 1.790,02 e R² médio de 0,5411, apresentando desempenho inferior ao modelo baseline nas três métricas consideradas.

Os resultados obtidos foram:

| Modelo | MAE médio | RMSE médio | R² médio |
|---|---:|---:|---:|
| XGBoost Baseline | R$ 1.123,79 | **R$ 1.779,48** | **0,5460** |
| XGBoost + High Standard | **R$ 1.122,70** | R$ 1.784,63 | 0,5435 |
| XGBoost + High Standard Type | R$ 1.124,56 | R$ 1.790,02 | 0,5411 |

De maneira geral, os experimentos indicam que as features `high_standard` e `high_standard_type` não proporcionaram uma melhoria relevante em relação às características originais utilizadas pelo modelo. A pequena redução do MAE observada no modelo com `high_standard` não foi acompanhada por melhorias no RMSE ou no R² e, portanto, não representa uma evidência suficiente de ganho efetivo de desempenho.

O resultado também sugere que o `XGBoost` já é capaz de capturar, a partir das variáveis `area`, `bedrooms`, `garage` e `type`, parte das relações não lineares utilizadas na construção dessas novas variáveis. Dessa forma, a transformação dessas informações em variáveis binárias de classificação de alto padrão pode não adicionar informação suficiente ao modelo para justificar sua inclusão.

Além disso, a análise dos erros realizada no notebook `05_modeling.ipynb` indicou que uma parcela importante dos maiores erros estava concentrada em imóveis com valores de aluguel elevados. Esses resultados motivaram a criação das features relacionadas ao padrão dos imóveis. Entretanto, os experimentos realizados neste notebook não demonstraram que essas variáveis foram suficientes para melhorar a capacidade do modelo de prever esse segmento da distribuição.

Os resultados sugerem que as limitações observadas estão possivelmente relacionadas não apenas ao algoritmo utilizado, mas também à disponibilidade e à capacidade explicativa das variáveis presentes no dataset. Características como localização detalhada, infraestrutura da região, proximidade de transporte, estado de conservação, mobiliário e outros atributos qualitativos do imóvel poderiam fornecer informações adicionais relevantes para a previsão do valor do aluguel.

Diante dos resultados obtidos, o `XGBoost Baseline` foi mantido como o modelo de referência final deste projeto, apresentando MAE médio de aproximadamente R$ 1.124, RMSE médio de aproximadamente R$ 1.779 e R² médio de aproximadamente 0,546 na validação cruzada. Assim, os experimentos realizados demonstraram que as técnicas de Feature Engineering desenvolvidas neste projeto não produziram ganhos relevantes de desempenho nas condições atuais do dataset. Em vez de realizar novas otimizações de forma indiscriminada, optou-se por encerrar o projeto reconhecendo as limitações dos dados disponíveis e mantendo o modelo baseline como referência.

Essa decisão reforça a importância de avaliar criticamente os resultados de um projeto de Machine Learning. A melhoria de desempenho não deve ser buscada apenas por meio de alterações no modelo ou pela criação de novas variáveis, especialmente quando essas alterações não apresentam ganhos consistentes ou podem introduzir riscos metodológicos, como data leakage.